In [ ]:
# === Required Imports ===
import tkinter as tk
from tkinter import messagebox, filedialog
import pandas as pd
import pickle

# === Load Trained Model ===
with open("GBM_SMA_model.pkl", "rb") as f:
    model_loaded = pickle.load(f)

# === Main Window ===
root = tk.Tk()
root.title("FOS Prediction | Hybrid GBM–SMA Model")
root.geometry("740x560")
root.resizable(False, False)

# === Canvas ===
canvas = tk.Canvas(root, width=740, height=560, bg="#eaf2f8")
canvas.pack()

# === Header ===
canvas.create_text(
    370, 30,
    text="Factor of Safety (FOS) Prediction using Hybrid GBM–SMA Model",
    font=("Times New Roman", 15, "bold"),
    fill="#1f3a5f"
)

canvas.create_text(
    370, 55,
    text="Decision Support Tool for 3D Slope Stability Analysis",
    font=("Times New Roman", 10, "italic"),
    fill="#4d4d4d"
)

# === Input Section ===
canvas.create_text(60, 100, text="Input Parameters", font=("Times New Roman", 13, "bold"), anchor="w")

labels = ["c (kPa)", "ϕ (°)", "γ (kN/m³)", "rᵤ (–)", "kₑ (–)"]
columns = ['c', 'Ø', 'ϒ', 'rᵤ', 'kₑ']
entries = []

y_pos = 140
for lbl in labels:
    label = tk.Label(root, text=lbl, font=("Times New Roman", 11), bg="#eaf2f8")
    canvas.create_window(60, y_pos, anchor="w", window=label)

    entry = tk.Entry(root, width=18, font=("Times New Roman", 11), relief="groove")
    canvas.create_window(360, y_pos, anchor="w", window=entry)
    entries.append(entry)

    y_pos += 35

# === Output Section ===
canvas.create_text(60, y_pos + 20, text="Predicted Output", font=("Times New Roman", 13, "bold"), anchor="w")

output_var = tk.StringVar(value="—")

output_box = tk.Label(
    root,
    textvariable=output_var,
    font=("Times New Roman", 13, "bold"),
    bg="white",
    fg="#1f3a5f",
    relief="solid",
    width=18,
    height=1
)
canvas.create_window(360, y_pos + 20, anchor="w", window=output_box)

# === Functions ===
def predict():
    try:
        input_values = [float(entry.get()) for entry in entries]
    except ValueError:
        messagebox.showerror("Input Error", "Please enter valid numerical values.")
        return

    df_input = pd.DataFrame([input_values], columns=columns)
    prediction = model_loaded.predict(df_input)

    output_var.set(f"{prediction[0]:.3f}")

def clear_all():
    for entry in entries:
        entry.delete(0, tk.END)
    output_var.set("—")

def export_csv():
    if output_var.get() == "—":
        messagebox.showwarning("Export Error", "Please generate a prediction first.")
        return

    data = {columns[i]: entries[i].get() for i in range(len(columns))}
    data["Predicted_FOS"] = output_var.get()

    df = pd.DataFrame([data])

    file_path = filedialog.asksaveasfilename(
        defaultextension=".csv",
        filetypes=[("CSV files", "*.csv")]
    )

    if file_path:
        df.to_csv(file_path, index=False)
        messagebox.showinfo("Success", "Results exported successfully.")

# === Buttons ===
btn_y = y_pos + 70

predict_btn = tk.Button(
    root, text="Predict FOS",
    command=predict,
    font=("Times New Roman", 12, "bold"),
    bg="#1f618d", fg="white",
    width=14
)
canvas.create_window(200, btn_y, window=predict_btn)

clear_btn = tk.Button(
    root, text="Clear All",
    command=clear_all,
    font=("Times New Roman", 12, "bold"),
    bg="#d35400", fg="white",
    width=14
)
canvas.create_window(370, btn_y, window=clear_btn)

export_btn = tk.Button(
    root, text="Export to CSV",
    command=export_csv,
    font=("Times New Roman", 12, "bold"),
    bg="#117a65", fg="white",
    width=14
)
canvas.create_window(540, btn_y, window=export_btn)

# === Footer ===
canvas.create_text(
    370, 530,
    text="Hybrid Gradient Boosting Machine with Slime Mould Algorithm (GBM–SMA)",
    font=("Times New Roman", 9),
    fill="#555555"
)

# === Run ===
root.mainloop()
